In [0]:
# 1. Read the data
# TODO: Replace the path below with your actual file path (e.g., "dbfs:/...", "abfss:/...", etc.)
df = spark.read.format("csv").option("header", "true").load("/Volumes/hdfc_ergo/dummy_data/raw/Dim_Members.csv")

new_cols = [c.lower().replace(" ","_").replace("-","_") for c in df.columns]
df = df.toDF(*new_cols )
# 2. Add Audit Columns (Unity Catalog Compliant)
from pyspark.sql.functions import current_timestamp

df = df.withColumn("_ingested_at", current_timestamp()) \
       .withColumn("_source_file", df["_metadata.file_path"])

# 3. Display to verify
display(df.limit(5))

# 4. Write to Bronze schema 
df.write.mode("overwrite").saveAsTable("hdfc_ergo.hdfc_bronze.dim_members")